In [1]:
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

/home/ethanlmines/blue_dir/repos/cluster-podcast-transcription/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "meta-llama/Llama-3.1-8B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 2363.16it/s]


In [5]:
orig_topics = ["politics", "business", "sports", "entertainment"]
standard_messages = [
    {"role": "system", "content": "Only answer in comma-delimited lists with no quotations:\na,b,c."},
]

In [4]:
from tqdm import tqdm
import csv
import glob
import os
texts = []
data_dir = "resegment_out/"

# Sample one episode from each show
show_dirs = [d_path for d_path in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d_path)]
for show_dir in tqdm(show_dirs, desc="Processing shows"):
    for csv_path in glob.glob(os.path.join(show_dir, "*.csv")):
        with open(csv_path, 'r') as r:
            texts.append("".join(row['text'] for row in csv.DictReader(r)))

Processing shows: 100%|██████████| 100/100 [00:47<00:00,  2.10it/s]


In [8]:
import re
output_pattern = re.compile('[^,\"\']+')

In [19]:
import xgrammar as xgr
tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer)
grammar = xgr.GrammarCompiler(tokenizer_info).compile_grammar(r"""
root ::=  () | topic ("," topic){0,2}
topic ::= [a-z][a-z ]{0,29}
""")

In [ ]:
cur_topics = sorted(orig_topics)

user_prefix = f"""
You are tasked with performing topic modeling over podcast transcripts.
Here are the topics you've discovered so far:
{','.join(cur_topics)}

Given the following podcast transcript, generate a list of no more than 3 new comma-delimited topics:
"""

for text in texts:
    tokenized_prompt = tokenizer.apply_chat_template(standard_messages + [
        {"role": "user", "content": user_prefix + "\n" + text}
    ],
    tokenize=True, add_generation_prompt=True, return_tensors='pt').to(model.device)
    output = model.generate(**tokenized_prompt, logits_processor=[xgr.contrib.hf.LogitsProcessor(grammar)], max_length=1024)
    input_length = tokenized_prompt['input_ids'].shape[-1]
    decoded = tokenizer.decode(output[0][input_length:], skip_special_tokens=True)
    new_topics = output_pattern.findall(decoded)
    cur_topics = sorted(set(cur_topics + new_topics))
    print(cur_topics)


/home/ethanlmines/blue_dir/repos/cluster-podcast-transcription/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=12923) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


['business', 'entertainment', 'inner strength', 'politics', 'psychedelics', 'spirituality', 'sports']


/home/ethanlmines/blue_dir/repos/cluster-podcast-transcription/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=14974) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


['business', 'entertainment', 'inner strength', 'politics', 'psychedelics', 'religion', 'spirituality', 'sports']


/home/ethanlmines/blue_dir/repos/cluster-podcast-transcription/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=10470) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


...

['building muscle', 'business', 'ceremony', 'compound interest', 'emotional regulation', 'entertainment', 'financial literacy money rehab', 'initiation', 'inner strength', 'investing', 'manifestation', 'new topics', 'politics', 'politics sports true love self', 'psychedelics', 'religion', 'self awareness', 'sexuality', 'spirituality', 'spirituality self improvement', 'sports', 'trauma response']


/home/ethanlmines/blue_dir/repos/cluster-podcast-transcription/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=13914) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


['building muscle', 'business', 'ceremony', 'compound interest', 'divine feminine', 'emotional healing', 'emotional regulation', 'entertainment', 'financial literacy money rehab', 'initiation', 'inner strength', 'investing', 'manifestation', 'new topics', 'politics', 'politics sports true love self', 'psychedelics', 'religion', 'self awareness', 'self discovery', 'sexuality', 'spirituality', 'spirituality self improvement', 'sports', 'trauma response']


/home/ethanlmines/blue_dir/repos/cluster-podcast-transcription/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=17791) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


KeyboardInterrupt: 